# GraphRAG Tutorial: From Basic RAG to Knowledge Graph-Enhanced Retrieval

The notebook is created thanks to PhD student Faezeh Safari at the Department of Computer Science, University of Exeter.

**Topic:** Graph Retrieval-Augmented Generation (GraphRAG)  


---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. ✅ Understand the limitations of traditional RAG systems
2. ✅ Implement document chunking strategies
3. ✅ Build TF-IDF vectorizers from scratch
4. ✅ Extract entities and relationships from text
5. ✅ Construct knowledge graphs using adjacency lists
6. ✅ Implement graph-based retrieval mechanisms
7. ✅ Compare and contrast plain RAG vs. GraphRAG performance

---

## Table of Contents

1. [Introduction to RAG and GraphRAG](#section1)
2. [Setup and Sample Data](#section2)
3. [Document Chunking](#section3)
4. [TF-IDF Vectorization](#section4)
5. [Plain RAG Implementation](#section5)
6. [Entity Extraction](#section6)
7. [Graph Construction](#section7)
8. [GraphRAG Implementation](#section8)
9. [Comparison: Plain RAG vs GraphRAG](#section9)
10. [Exercises and Extensions](#section10)

<a id='section1'></a>
## 1. Introduction to RAG and GraphRAG

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances language models by:
1. Retrieving relevant documents from a knowledge base
2. Using retrieved context to generate more accurate responses

### Limitations of Plain RAG

- **Isolated chunks**: Documents are split into independent chunks
- **No relationship modeling**: Connections between entities are lost
- **Limited context**: May miss related information in different chunks
- **Semantic gaps**: Similar concepts with different terminology aren't connected

### What is GraphRAG?

**GraphRAG** extends RAG by:
1. Extracting **entities** (people, places, concepts) from documents
2. Identifying **relationships** between entities
3. Building a **knowledge graph** structure
4. Using **graph traversal** for richer retrieval

### Key Advantages

✨ **Multi-hop reasoning**: Connect related information across documents  
✨ **Entity-centric retrieval**: Find all information about specific entities  
✨ **Relationship awareness**: Understand how concepts relate to each other  
✨ **Better context**: Retrieve semantically connected chunks

---

<a id='section2'></a>
## 2. Setup and Sample Data

We'll use only Python standard library and NumPy to keep things transparent and educational.

In [1]:
# Import required libraries
import re
import math
import numpy as np
from collections import defaultdict, Counter
from typing import List, Dict, Set, Tuple
import json

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


### Creating Sample Documents

We'll create a small corpus about AI researchers and their contributions. This domain is perfect for demonstrating GraphRAG because it has:
- Clear entities (people, institutions, concepts)
- Rich relationships (worked with, developed, studied at)
- Multi-hop connections (A worked with B, B developed C)

In [2]:
# Sample corpus about AI researchers
documents = [
    """Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist.
    He is known for his work on artificial neural networks. Hinton worked at the University of Toronto
    and Google. He received the Turing Award in 2018 along with Yoshua Bengio and Yann LeCun.""",

    """Yoshua Bengio is a Canadian computer scientist working in artificial intelligence.
    He is a professor at the University of Montreal. Bengio pioneered deep learning research
    and received the Turing Award in 2018. He focuses on neural networks and their applications.""",

    """Yann LeCun is a French computer scientist working in machine learning and computer vision.
    He is a professor at New York University and works at Meta AI Research. LeCun developed
    convolutional neural networks and received the Turing Award in 2018.""",

    """The Turing Award is often called the Nobel Prize of computing. It was established in 1966
    by the Association for Computing Machinery. The award recognizes contributions of lasting
    importance to computing. Winners receive a prize of one million dollars.""",

    """Deep learning is a subset of machine learning based on artificial neural networks.
    The technique has been applied to computer vision, natural language processing, and speech recognition.
    Geoffrey Hinton, Yoshua Bengio, and Yann LeCun are considered pioneers of deep learning.""",

    """Convolutional neural networks are a type of deep neural network commonly used for image processing.
    Yann LeCun developed the architecture while working on handwritten digit recognition.
    CNNs have revolutionized computer vision and are used in facial recognition and autonomous vehicles.""",

    """The University of Toronto has been a major center for artificial intelligence research.
    Geoffrey Hinton established a research group there focusing on neural networks.
    Many breakthrough discoveries in deep learning came from Toronto researchers.""",

    """Natural language processing enables computers to understand and generate human language.
    Modern NLP relies heavily on deep learning and neural networks. Applications include
    machine translation, sentiment analysis, and chatbots like ChatGPT."""
]

print(f"📚 Loaded {len(documents)} documents")
print(f"📊 Average document length: {sum(len(doc) for doc in documents) / len(documents):.0f} characters")
print("\n📄 Sample document:")
print(documents[0][:200] + "...")

📚 Loaded 8 documents
📊 Average document length: 270 characters

📄 Sample document:
Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. 
    He is known for his work on artificial neural networks. Hinton worked at the University of Toronto 
    and Go...


<a id='section3'></a>
## 3. Document Chunking

### Why Chunk Documents?

Large documents need to be split into smaller chunks because:
- **Computational efficiency**: Smaller units are faster to process
- **Retrieval precision**: More granular matching with queries
- **Context windows**: LLMs have limited input sizes

### Chunking Strategy

We'll implement **sentence-based chunking** with overlap:
- Split on sentence boundaries (periods, question marks, exclamation points)
- Group sentences into chunks of specified size
- Add overlap between chunks to maintain context

In [3]:
def chunk_document(text: str, chunk_size: int = 2, overlap: int = 1) -> List[str]:
    """
    Split document into overlapping chunks based on sentences.

    Parameters:
    -----------
    text : str
        The document to chunk
    chunk_size : int
        Number of sentences per chunk
    overlap : int
        Number of sentences to overlap between chunks

    Returns:
    --------
    List[str]
        List of text chunks
    """
    # Split into sentences using regex
    # This pattern splits on .!? followed by space or end of string
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    # Remove empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []

    # Create overlapping chunks
    for i in range(0, len(sentences), chunk_size - overlap):
        chunk = ' '.join(sentences[i:i + chunk_size])
        if chunk:  # Only add non-empty chunks
            chunks.append(chunk)

        # Stop if we've reached the end
        if i + chunk_size >= len(sentences):
            break

    return chunks

# Test the chunking function
print("🔪 Testing chunking on first document:\n")
test_chunks = chunk_document(documents[0], chunk_size=2, overlap=1)
for i, chunk in enumerate(test_chunks):
    print(f"Chunk {i+1}: {chunk}")
    print()

🔪 Testing chunking on first document:

Chunk 1: Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. He is known for his work on artificial neural networks.

Chunk 2: He is known for his work on artificial neural networks. Hinton worked at the University of Toronto 
    and Google.

Chunk 3: Hinton worked at the University of Toronto 
    and Google. He received the Turing Award in 2018 along with Yoshua Bengio and Yann LeCun.



In [4]:
# Chunk all documents
all_chunks = []
chunk_to_doc = []  # Track which document each chunk belongs to

for doc_id, doc in enumerate(documents):
    chunks = chunk_document(doc, chunk_size=2, overlap=1)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_id] * len(chunks))

print(f"✂️ Created {len(all_chunks)} chunks from {len(documents)} documents")
print(f"📊 Average chunks per document: {len(all_chunks) / len(documents):.1f}")
print(f"\n📝 Sample chunks:")
for i in range(min(3, len(all_chunks))):
    print(f"\n[Chunk {i+1}] (from doc {chunk_to_doc[i]})")
    print(all_chunks[i])

✂️ Created 19 chunks from 8 documents
📊 Average chunks per document: 2.4

📝 Sample chunks:

[Chunk 1] (from doc 0)
Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. He is known for his work on artificial neural networks.

[Chunk 2] (from doc 0)
He is known for his work on artificial neural networks. Hinton worked at the University of Toronto 
    and Google.

[Chunk 3] (from doc 0)
Hinton worked at the University of Toronto 
    and Google. He received the Turing Award in 2018 along with Yoshua Bengio and Yann LeCun.


<a id='section4'></a>
## 4. TF-IDF Vectorization

### What is TF-IDF?

**TF-IDF (Term Frequency-Inverse Document Frequency)** is a numerical statistic that reflects how important a word is to a document in a collection.

### Mathematical Definition

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Where:
- **TF (Term Frequency)**: How often term $t$ appears in document $d$
  $$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total terms in } d}$$

- **IDF (Inverse Document Frequency)**: How rare the term is across all documents
  $$\text{IDF}(t) = \log\left(\frac{\text{total documents}}{\text{documents containing } t}\right)$$

### Intuition

- **High TF-IDF**: Word appears frequently in this document but rarely in others → Important!
- **Low TF-IDF**: Word is either rare in this document or common everywhere → Less distinctive

In [5]:
class TfidfVectorizer:
    """
    A simple TF-IDF vectorizer built from scratch.
    """

    def __init__(self):
        self.vocabulary = {}  # word -> index mapping
        self.idf_values = {}  # word -> IDF score
        self.num_documents = 0

    def tokenize(self, text: str) -> List[str]:
        """
        Convert text to lowercase tokens.
        """
        # Convert to lowercase and extract words
        tokens = re.findall(r'\b[a-z]+\b', text.lower())
        return tokens

    def fit(self, documents: List[str]):
        """
        Learn vocabulary and IDF values from documents.
        """
        self.num_documents = len(documents)

        # Count document frequency for each word
        doc_frequency = defaultdict(int)

        for doc in documents:
            tokens = self.tokenize(doc)
            unique_tokens = set(tokens)

            for token in unique_tokens:
                doc_frequency[token] += 1

        # Build vocabulary and calculate IDF
        for idx, (word, df) in enumerate(sorted(doc_frequency.items())):
            self.vocabulary[word] = idx
            # IDF formula: log(N / df)
            self.idf_values[word] = math.log(self.num_documents / df)

        return self

    def transform(self, documents: List[str]) -> np.ndarray:
        """
        Transform documents to TF-IDF vectors.

        Returns:
        --------
        np.ndarray of shape (n_documents, n_features)
        """
        n_docs = len(documents)
        n_features = len(self.vocabulary)

        # Initialize TF-IDF matrix
        tfidf_matrix = np.zeros((n_docs, n_features))

        for doc_idx, doc in enumerate(documents):
            tokens = self.tokenize(doc)
            token_counts = Counter(tokens)
            total_tokens = len(tokens)

            if total_tokens == 0:
                continue

            for token, count in token_counts.items():
                if token in self.vocabulary:
                    # TF: term frequency
                    tf = count / total_tokens
                    # IDF: inverse document frequency
                    idf = self.idf_values[token]
                    # TF-IDF score
                    word_idx = self.vocabulary[token]
                    tfidf_matrix[doc_idx, word_idx] = tf * idf

        # Normalize vectors to unit length (L2 normalization)
        norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1  # Avoid division by zero
        tfidf_matrix = tfidf_matrix / norms

        return tfidf_matrix

    def fit_transform(self, documents: List[str]) -> np.ndarray:
        """
        Fit and transform in one step.
        """
        self.fit(documents)
        return self.transform(documents)

print("✅ TF-IDF Vectorizer class defined")

✅ TF-IDF Vectorizer class defined


In [6]:
# Create and fit the vectorizer
vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(all_chunks)

print(f"📊 TF-IDF Matrix Shape: {chunk_vectors.shape}")
print(f"   - {chunk_vectors.shape[0]} chunks (documents)")
print(f"   - {chunk_vectors.shape[1]} unique terms (features)")
print(f"\n📚 Vocabulary size: {len(vectorizer.vocabulary)}")
print(f"\n🔤 Sample vocabulary (first 20 words):")
print(list(vectorizer.vocabulary.keys())[:20])

📊 TF-IDF Matrix Shape: (19, 135)
   - 19 chunks (documents)
   - 135 unique terms (features)

📚 Vocabulary size: 135

🔤 Sample vocabulary (first 20 words):
['a', 'ai', 'along', 'analysis', 'and', 'applications', 'applied', 'architecture', 'are', 'artificial', 'association', 'at', 'autonomous', 'award', 'based', 'been', 'bengio', 'breakthrough', 'british', 'by']


In [7]:
# Analyze TF-IDF scores for a sample chunk
sample_chunk_idx = 0
sample_vector = chunk_vectors[sample_chunk_idx]

# Get top words by TF-IDF score
word_scores = [(word, sample_vector[idx]) for word, idx in vectorizer.vocabulary.items()]
word_scores.sort(key=lambda x: x[1], reverse=True)

print(f"📄 Analyzing chunk: '{all_chunks[sample_chunk_idx][:100]}...'\n")
print("🏆 Top 10 words by TF-IDF score:")
for word, score in word_scores[:10]:
    if score > 0:
        print(f"   {word:20s} {score:.4f}")

📄 Analyzing chunk: 'Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. He is known for...'

🏆 Top 10 words by TF-IDF score:
   british              0.3695
   cognitive            0.3695
   psychologist         0.3695
   canadian             0.2825
   his                  0.2825
   known                0.2825
   work                 0.2825
   scientist            0.2317
   is                   0.2171
   geoffrey             0.1956


<a id='section5'></a>
## 5. Plain RAG Implementation

### How Plain RAG Works

1. **Query Processing**: Convert user query to TF-IDF vector
2. **Similarity Calculation**: Compute cosine similarity with all chunks
3. **Retrieval**: Return top-k most similar chunks
4. **Generation**: Use retrieved chunks as context (simulated here)

### Cosine Similarity

$$\text{similarity}(A, B) = \frac{A \cdot B}{||A|| \times ||B||} = \frac{\sum_{i} A_i B_i}{\sqrt{\sum_{i} A_i^2} \times \sqrt{\sum_{i} B_i^2}}$$

Since our vectors are already normalized, this simplifies to dot product!

In [8]:
class PlainRAG:
    """
    Simple Retrieval-Augmented Generation using TF-IDF and cosine similarity.
    """

    def __init__(self, chunks: List[str], vectorizer: TfidfVectorizer, chunk_vectors: np.ndarray):
        self.chunks = chunks
        self.vectorizer = vectorizer
        self.chunk_vectors = chunk_vectors

    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """
        Retrieve top-k most relevant chunks for a query.

        Parameters:
        -----------
        query : str
            The search query
        top_k : int
            Number of chunks to retrieve

        Returns:
        --------
        List[Tuple[str, float]]
            List of (chunk_text, similarity_score) tuples
        """
        # Convert query to TF-IDF vector
        query_vector = self.vectorizer.transform([query])[0]

        # Calculate cosine similarity with all chunks
        # Since vectors are normalized, cosine similarity = dot product
        similarities = np.dot(self.chunk_vectors, query_vector)

        # Get top-k indices
        top_indices = np.argsort(similarities)[::-1][:top_k]

        # Return chunks with scores
        results = [
            (self.chunks[idx], float(similarities[idx]))
            for idx in top_indices
        ]

        return results

    def answer(self, query: str, top_k: int = 3) -> Dict:
        """
        Retrieve relevant chunks and format as answer context.
        """
        retrieved = self.retrieve(query, top_k)

        # In a real RAG system, this context would be sent to an LLM
        context = "\n\n".join([chunk for chunk, _ in retrieved])

        return {
            "query": query,
            "retrieved_chunks": retrieved,
            "context": context
        }

# Initialize Plain RAG system
plain_rag = PlainRAG(all_chunks, vectorizer, chunk_vectors)
print("✅ Plain RAG system initialized")

✅ Plain RAG system initialized


In [9]:
# Test Plain RAG with sample queries
test_queries = [
    "Who won the Turing Award?",
    "What is deep learning?",
    "Tell me about convolutional neural networks"
]

print("🔍 Testing Plain RAG System\n")
print("=" * 80)

for query in test_queries:
    print(f"\n❓ Query: {query}\n")
    result = plain_rag.answer(query, top_k=3)

    print("📚 Retrieved Chunks:")
    for i, (chunk, score) in enumerate(result['retrieved_chunks'], 1):
        print(f"\n  [{i}] Score: {score:.4f}")
        print(f"      {chunk[:150]}...")

    print("\n" + "=" * 80)

🔍 Testing Plain RAG System


❓ Query: Who won the Turing Award?

📚 Retrieved Chunks:

  [1] Score: 0.3312
      He is a professor at the University of Montreal. Bengio pioneered deep learning research 
    and received the Turing Award in 2018....

  [2] Score: 0.2659
      Bengio pioneered deep learning research 
    and received the Turing Award in 2018. He focuses on neural networks and their applications....

  [3] Score: 0.2481
      Hinton worked at the University of Toronto 
    and Google. He received the Turing Award in 2018 along with Yoshua Bengio and Yann LeCun....


❓ Query: What is deep learning?

📚 Retrieved Chunks:

  [1] Score: 0.2836
      He is a professor at the University of Montreal. Bengio pioneered deep learning research 
    and received the Turing Award in 2018....

  [2] Score: 0.2414
      Deep learning is a subset of machine learning based on artificial neural networks. The technique has been applied to computer vision, natural language...

  [3] Score: 0.1

<a id='section6'></a>
## 6. Entity Extraction

### What are Entities?

**Entities** are important objects mentioned in text:
- **People**: Geoffrey Hinton, Yoshua Bengio
- **Organizations**: University of Toronto, Google
- **Concepts**: Deep learning, neural networks
- **Awards**: Turing Award

### Simple Entity Extraction Strategy

For educational purposes, we'll use a simple approach:
1. **Capitalized phrases**: Multi-word phrases that start with capitals
2. **Named patterns**: Match against known entity types
3. **Frequency filtering**: Keep entities that appear multiple times

**Note**: Production systems use NER models like spaCy or BERT-based extractors.

In [10]:
class EntityExtractor:
    """
    Simple rule-based entity extractor for educational purposes.
    """

    def __init__(self):
        # Patterns that indicate entity types
        self.entity_patterns = {
            'PERSON': r'\b[A-Z][a-z]+ [A-Z][a-z]+\b',  # First Last
            'ORGANIZATION': r'\b(?:University of [A-Z][a-z]+|[A-Z][a-z]+ University)\b',
            'AWARD': r'\b(?:Turing Award|Nobel Prize)\b',
        }

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """
        Extract entities from text using simple patterns.

        Returns:
        --------
        Dict[str, List[str]]
            Dictionary mapping entity types to lists of entities
        """
        entities = defaultdict(list)

        # Extract using patterns
        for entity_type, pattern in self.entity_patterns.items():
            matches = re.findall(pattern, text)
            entities[entity_type].extend(matches)

        # Also extract capitalized phrases (2-4 words)
        capitalized_phrases = re.findall(
            r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3}\b',
            text
        )

        # Filter to get potential concepts
        for phrase in capitalized_phrases:
            # Skip if already captured
            if not any(phrase in ents for ents in entities.values()):
                # Check if it's a concept (contains certain keywords)
                if any(word in phrase.lower() for word in
                       ['learning', 'network', 'processing', 'intelligence', 'vision']):
                    entities['CONCEPT'].append(phrase)

        # Remove duplicates while preserving order
        for entity_type in entities:
            seen = set()
            unique = []
            for entity in entities[entity_type]:
                if entity not in seen:
                    seen.add(entity)
                    unique.append(entity)
            entities[entity_type] = unique

        return dict(entities)

    def extract_from_chunks(self, chunks: List[str]) -> Dict[int, Dict[str, List[str]]]:
        """
        Extract entities from multiple chunks.

        Returns:
        --------
        Dict[int, Dict[str, List[str]]]
            Mapping from chunk_id to entities found
        """
        chunk_entities = {}

        for chunk_id, chunk in enumerate(chunks):
            entities = self.extract_entities(chunk)
            if entities:  # Only store if entities were found
                chunk_entities[chunk_id] = entities

        return chunk_entities

# Initialize entity extractor
entity_extractor = EntityExtractor()
print("✅ Entity Extractor initialized")

✅ Entity Extractor initialized


In [11]:
# Test entity extraction on a sample chunk
sample_text = documents[0]
print("📄 Sample Text:")
print(sample_text)
print("\n" + "=" * 80)
print("\n🏷️ Extracted Entities:\n")

entities = entity_extractor.extract_entities(sample_text)
for entity_type, entity_list in entities.items():
    print(f"  {entity_type}:")
    for entity in entity_list:
        print(f"    - {entity}")

📄 Sample Text:
Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. 
    He is known for his work on artificial neural networks. Hinton worked at the University of Toronto 
    and Google. He received the Turing Award in 2018 along with Yoshua Bengio and Yann LeCun.


🏷️ Extracted Entities:

  PERSON:
    - Geoffrey Hinton
    - Turing Award
    - Yoshua Bengio
  ORGANIZATION:
    - University of Toronto
  AWARD:
    - Turing Award


In [12]:
# Extract entities from all chunks
chunk_entities = entity_extractor.extract_from_chunks(all_chunks)

print(f"📊 Entity Extraction Statistics:")
print(f"   - Chunks with entities: {len(chunk_entities)} / {len(all_chunks)}")

# Count total entities by type
entity_type_counts = defaultdict(int)
all_unique_entities = defaultdict(set)

for chunk_id, entities in chunk_entities.items():
    for entity_type, entity_list in entities.items():
        entity_type_counts[entity_type] += len(entity_list)
        all_unique_entities[entity_type].update(entity_list)

print(f"\n📈 Entity Type Counts:")
for entity_type, count in entity_type_counts.items():
    unique_count = len(all_unique_entities[entity_type])
    print(f"   {entity_type:15s}: {count:3d} total, {unique_count:3d} unique")

print(f"\n🔤 Sample Unique Entities:")
for entity_type, entity_set in all_unique_entities.items():
    print(f"   {entity_type}: {list(entity_set)[:5]}")

📊 Entity Extraction Statistics:
   - Chunks with entities: 19 / 19

📈 Entity Type Counts:
   PERSON         :  18 total,   8 unique
   ORGANIZATION   :   7 total,   4 unique
   AWARD          :   6 total,   2 unique

🔤 Sample Unique Entities:
   PERSON: ['Yoshua Bengio', 'Nobel Prize', 'The University', 'Computing Machinery', 'Geoffrey Hinton']
   ORGANIZATION: ['The University', 'University of Toronto', 'York University', 'University of Montreal']
   AWARD: ['Nobel Prize', 'Turing Award']


<a id='section7'></a>
## 7. Graph Construction

### What is a Knowledge Graph?

A **knowledge graph** represents entities and their relationships:
- **Nodes**: Entities (people, concepts, organizations)
- **Edges**: Relationships between entities (worked_at, developed, won)

### Graph Representation: Adjacency List

We'll use an **adjacency list** structure:
```python
{
    "Geoffrey Hinton": {
        "University of Toronto": {"relation": "worked_at", "weight": 1},
        "Turing Award": {"relation": "won", "weight": 1}
    }
}
```

### Relationship Types

- **Co-occurrence**: Entities appearing in the same chunk
- **Chunk membership**: Entity belongs to chunk
- **Document membership**: Entity belongs to document

In [13]:
class KnowledgeGraph:
    """
    Simple knowledge graph using adjacency list representation.
    """

    def __init__(self):
        # Graph structure: {entity: {neighbor: {relation: str, weight: float}}}
        self.graph = defaultdict(lambda: defaultdict(dict))

        # Entity metadata
        self.entity_to_chunks = defaultdict(set)  # entity -> set of chunk_ids
        self.entity_types = {}  # entity -> type

        # Chunk metadata
        self.chunk_to_entities = defaultdict(set)  # chunk_id -> set of entities

    def add_edge(self, entity1: str, entity2: str, relation: str = "related_to", weight: float = 1.0):
        """
        Add an edge between two entities.
        """
        # Add bidirectional edge
        self.graph[entity1][entity2] = {"relation": relation, "weight": weight}
        self.graph[entity2][entity1] = {"relation": relation, "weight": weight}

    def add_entity(self, entity: str, entity_type: str, chunk_id: int):
        """
        Add an entity with its metadata.
        """
        self.entity_types[entity] = entity_type
        self.entity_to_chunks[entity].add(chunk_id)
        self.chunk_to_entities[chunk_id].add(entity)

    def build_from_chunks(self, chunk_entities: Dict[int, Dict[str, List[str]]]):
        """
        Build graph from extracted entities.

        Parameters:
        -----------
        chunk_entities : Dict[int, Dict[str, List[str]]]
            Mapping from chunk_id to entities
        """
        # First pass: Add all entities
        for chunk_id, entities_by_type in chunk_entities.items():
            for entity_type, entity_list in entities_by_type.items():
                for entity in entity_list:
                    self.add_entity(entity, entity_type, chunk_id)

        # Second pass: Create co-occurrence edges
        for chunk_id, entities_by_type in chunk_entities.items():
            # Flatten all entities in this chunk
            chunk_entities_flat = []
            for entity_list in entities_by_type.values():
                chunk_entities_flat.extend(entity_list)

            # Create edges between all pairs (co-occurrence)
            for i, entity1 in enumerate(chunk_entities_flat):
                for entity2 in chunk_entities_flat[i+1:]:
                    # Weight by number of co-occurrences
                    existing_weight = self.graph[entity1].get(entity2, {}).get("weight", 0)
                    self.add_edge(entity1, entity2, "co_occurs", existing_weight + 1)

    def get_neighbors(self, entity: str, max_depth: int = 1) -> Set[str]:
        """
        Get all neighbors of an entity up to max_depth.

        This implements graph traversal (BFS).
        """
        if entity not in self.graph:
            return set()

        visited = {entity}
        current_level = {entity}

        for _ in range(max_depth):
            next_level = set()
            for node in current_level:
                if node in self.graph:
                    neighbors = set(self.graph[node].keys())
                    next_level.update(neighbors - visited)

            visited.update(next_level)
            current_level = next_level

            if not current_level:
                break

        return visited - {entity}  # Exclude the entity itself

    def get_stats(self) -> Dict:
        """
        Get graph statistics.
        """
        total_edges = sum(len(neighbors) for neighbors in self.graph.values()) // 2

        return {
            "num_entities": len(self.graph),
            "num_edges": total_edges,
            "avg_degree": 2 * total_edges / len(self.graph) if self.graph else 0,
            "entity_types": dict(Counter(self.entity_types.values()))
        }

# Build the knowledge graph
kg = KnowledgeGraph()
kg.build_from_chunks(chunk_entities)

print("✅ Knowledge Graph constructed")

✅ Knowledge Graph constructed


In [14]:
# Analyze the knowledge graph
stats = kg.get_stats()

print("📊 Knowledge Graph Statistics:")
print(f"   - Total Entities: {stats['num_entities']}")
print(f"   - Total Edges: {stats['num_edges']}")
print(f"   - Average Degree: {stats['avg_degree']:.2f}")
print(f"\n🏷️ Entity Type Distribution:")
for entity_type, count in stats['entity_types'].items():
    print(f"   {entity_type:15s}: {count}")

📊 Knowledge Graph Statistics:
   - Total Entities: 11
   - Total Edges: 17
   - Average Degree: 3.09

🏷️ Entity Type Distribution:
   PERSON         : 5
   ORGANIZATION   : 4
   AWARD          : 2


In [15]:
# Explore entity relationships
sample_entity = "Geoffrey Hinton"

print(f"🔍 Exploring entity: '{sample_entity}'\n")

if sample_entity in kg.graph:
    print(f"📍 Appears in {len(kg.entity_to_chunks[sample_entity])} chunks")
    print(f"\n🔗 Direct neighbors ({len(kg.graph[sample_entity])}):\n")

    # Sort neighbors by weight
    neighbors = sorted(
        kg.graph[sample_entity].items(),
        key=lambda x: x[1]['weight'],
        reverse=True
    )

    for neighbor, info in neighbors[:10]:
        print(f"   → {neighbor:30s} (weight: {info['weight']:.1f})")

    # Get 2-hop neighbors
    two_hop = kg.get_neighbors(sample_entity, max_depth=2)
    print(f"\n🌐 2-hop neighborhood: {len(two_hop)} entities")
else:
    print(f"Entity '{sample_entity}' not found in graph")

🔍 Exploring entity: 'Geoffrey Hinton'

📍 Appears in 4 chunks

🔗 Direct neighbors (2):

   → The University                 (weight: 2.0)
   → Yoshua Bengio                  (weight: 1.0)

🌐 2-hop neighborhood: 5 entities


<a id='section8'></a>
## 8. GraphRAG Implementation

### How GraphRAG Works

GraphRAG extends plain RAG with graph-based retrieval:

1. **Entity Recognition**: Extract entities from the query
2. **Graph Expansion**: Find related entities using graph traversal
3. **Chunk Retrieval**: Get chunks containing relevant entities
4. **Ranking**: Combine vector similarity with graph signals
5. **Context Assembly**: Return enriched context

### Key Innovation

Instead of just matching query terms, GraphRAG:
- Finds **related concepts** through the graph
- Retrieves **multi-hop information**
- Provides **richer context** by including connected entities

In [16]:
class GraphRAG:
    """
    Graph-enhanced Retrieval-Augmented Generation.
    """

    def __init__(
        self,
        chunks: List[str],
        vectorizer: TfidfVectorizer,
        chunk_vectors: np.ndarray,
        knowledge_graph: KnowledgeGraph,
        entity_extractor: EntityExtractor
    ):
        self.chunks = chunks
        self.vectorizer = vectorizer
        self.chunk_vectors = chunk_vectors
        self.kg = knowledge_graph
        self.entity_extractor = entity_extractor

    def retrieve(self, query: str, top_k: int = 3, graph_depth: int = 1) -> List[Tuple[str, float, Dict]]:
        """
        Retrieve chunks using both vector similarity and graph structure.

        Parameters:
        -----------
        query : str
            Search query
        top_k : int
            Number of chunks to retrieve
        graph_depth : int
            How many hops to traverse in the graph

        Returns:
        --------
        List[Tuple[str, float, Dict]]
            List of (chunk_text, score, metadata) tuples
        """
        # Step 1: Extract entities from query
        query_entities = self.entity_extractor.extract_entities(query)
        query_entities_flat = []
        for entity_list in query_entities.values():
            query_entities_flat.extend(entity_list)

        # Step 2: Expand entities using graph
        expanded_entities = set(query_entities_flat)
        for entity in query_entities_flat:
            neighbors = self.kg.get_neighbors(entity, max_depth=graph_depth)
            expanded_entities.update(neighbors)

        # Step 3: Get candidate chunks from entities
        candidate_chunks = set()
        for entity in expanded_entities:
            if entity in self.kg.entity_to_chunks:
                candidate_chunks.update(self.kg.entity_to_chunks[entity])

        # Step 4: Calculate vector similarity
        query_vector = self.vectorizer.transform([query])[0]
        similarities = np.dot(self.chunk_vectors, query_vector)

        # Step 5: Calculate graph-based scores
        graph_scores = np.zeros(len(self.chunks))

        for chunk_id in candidate_chunks:
            # Score based on number of relevant entities in chunk
            chunk_entities = self.kg.chunk_to_entities.get(chunk_id, set())
            relevant_entities = chunk_entities & expanded_entities

            # Bonus for query entities vs expanded entities
            query_entity_count = len(chunk_entities & set(query_entities_flat))
            expanded_entity_count = len(relevant_entities)

            graph_scores[chunk_id] = query_entity_count * 2 + expanded_entity_count

        # Normalize graph scores
        if graph_scores.max() > 0:
            graph_scores = graph_scores / graph_scores.max()

        # Step 6: Combine scores (weighted average)
        alpha = 0.6  # Weight for vector similarity
        beta = 0.4   # Weight for graph score

        combined_scores = alpha * similarities + beta * graph_scores

        # Step 7: Get top-k chunks
        top_indices = np.argsort(combined_scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            metadata = {
                "chunk_id": int(idx),
                "vector_score": float(similarities[idx]),
                "graph_score": float(graph_scores[idx]),
                "combined_score": float(combined_scores[idx]),
                "entities": list(self.kg.chunk_to_entities.get(idx, set()))
            }
            results.append((self.chunks[idx], float(combined_scores[idx]), metadata))

        return results

    def answer(self, query: str, top_k: int = 3, graph_depth: int = 1) -> Dict:
        """
        Retrieve relevant chunks and format as answer context.
        """
        retrieved = self.retrieve(query, top_k, graph_depth)

        # Extract query entities for metadata
        query_entities = self.entity_extractor.extract_entities(query)

        context = "\n\n".join([chunk for chunk, _, _ in retrieved])

        return {
            "query": query,
            "query_entities": query_entities,
            "retrieved_chunks": retrieved,
            "context": context
        }

# Initialize GraphRAG system
graph_rag = GraphRAG(
    chunks=all_chunks,
    vectorizer=vectorizer,
    chunk_vectors=chunk_vectors,
    knowledge_graph=kg,
    entity_extractor=entity_extractor
)

print("✅ GraphRAG system initialized")

✅ GraphRAG system initialized


In [17]:
# Test GraphRAG with sample queries
print("🔍 Testing GraphRAG System\n")
print("=" * 80)

for query in test_queries:
    print(f"\n❓ Query: {query}\n")
    result = graph_rag.answer(query, top_k=3, graph_depth=1)

    if result['query_entities']:
        print("🏷️ Detected Query Entities:")
        for entity_type, entities in result['query_entities'].items():
            print(f"   {entity_type}: {entities}")

    print("\n📚 Retrieved Chunks:")
    for i, (chunk, score, metadata) in enumerate(result['retrieved_chunks'], 1):
        print(f"\n  [{i}] Combined Score: {score:.4f}")
        print(f"      Vector: {metadata['vector_score']:.4f} | Graph: {metadata['graph_score']:.4f}")
        print(f"      Entities: {metadata['entities'][:3]}{'...' if len(metadata['entities']) > 3 else ''}")
        print(f"      {chunk[:120]}...")

    print("\n" + "=" * 80)

🔍 Testing GraphRAG System


❓ Query: Who won the Turing Award?

🏷️ Detected Query Entities:
   PERSON: ['Turing Award']
   ORGANIZATION: []
   AWARD: ['Turing Award']

📚 Retrieved Chunks:

  [1] Combined Score: 0.5219
      Vector: 0.2032 | Graph: 1.0000
      Entities: ['Nobel Prize', 'Turing Award', 'The Turing']...
      The Turing Award is often called the Nobel Prize of computing. It was established in 1966 
    by the Association for Co...

  [2] Combined Score: 0.4822
      Vector: 0.2481 | Graph: 0.8333
      Entities: ['Yoshua Bengio', 'University of Toronto', 'Turing Award']
      Hinton worked at the University of Toronto 
    and Google. He received the Turing Award in 2018 along with Yoshua Bengi...

  [3] Combined Score: 0.4711
      Vector: 0.2296 | Graph: 0.8333
      Entities: ['York University', 'New York', 'Turing Award']
      He is a professor at New York University and works at Meta AI Research. LeCun developed 
    convolutional neural networ...


❓ Query: What i

<a id='section9'></a>
## 9. Comparison: Plain RAG vs GraphRAG

Let's systematically compare the two approaches on various types of queries.

In [18]:
def compare_rag_systems(query: str, top_k: int = 3):
    """
    Compare Plain RAG and GraphRAG side-by-side.
    """
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}\n")

    # Get results from both systems
    plain_result = plain_rag.answer(query, top_k)
    graph_result = graph_rag.answer(query, top_k, graph_depth=1)

    # Display Plain RAG results
    print("📊 PLAIN RAG RESULTS:")
    print("-" * 80)
    for i, (chunk, score) in enumerate(plain_result['retrieved_chunks'], 1):
        print(f"\n[{i}] Score: {score:.4f}")
        print(f"    {chunk[:150]}...")

    # Display GraphRAG results
    print("\n\n📊 GraphRAG RESULTS:")
    print("-" * 80)
    if graph_result['query_entities']:
        print("\n🏷️ Query Entities Detected:")
        for entity_type, entities in graph_result['query_entities'].items():
            if entities:
                print(f"   {entity_type}: {entities}")

    for i, (chunk, score, metadata) in enumerate(graph_result['retrieved_chunks'], 1):
        print(f"\n[{i}] Combined: {score:.4f} (Vector: {metadata['vector_score']:.4f}, Graph: {metadata['graph_score']:.4f})")
        print(f"    Entities: {metadata['entities'][:3]}")
        print(f"    {chunk[:150]}...")

    # Analysis
    print("\n\n📈 ANALYSIS:")
    print("-" * 80)

    # Calculate overlap
    plain_chunks = {chunk for chunk, _ in plain_result['retrieved_chunks']}
    graph_chunks = {chunk for chunk, _, _ in graph_result['retrieved_chunks']}
    overlap = len(plain_chunks & graph_chunks)

    print(f"Chunk Overlap: {overlap}/{top_k} chunks are the same")

    if overlap < top_k:
        print("\n✨ GraphRAG found different chunks due to:")
        print("   - Entity relationship awareness")
        print("   - Graph-based context expansion")

    print("\n")

In [19]:
# Comparison Test 1: Entity-centric query
compare_rag_systems("Who won the Turing Award?")


Query: Who won the Turing Award?

📊 PLAIN RAG RESULTS:
--------------------------------------------------------------------------------

[1] Score: 0.3312
    He is a professor at the University of Montreal. Bengio pioneered deep learning research 
    and received the Turing Award in 2018....

[2] Score: 0.2659
    Bengio pioneered deep learning research 
    and received the Turing Award in 2018. He focuses on neural networks and their applications....

[3] Score: 0.2481
    Hinton worked at the University of Toronto 
    and Google. He received the Turing Award in 2018 along with Yoshua Bengio and Yann LeCun....


📊 GraphRAG RESULTS:
--------------------------------------------------------------------------------

🏷️ Query Entities Detected:
   PERSON: ['Turing Award']
   AWARD: ['Turing Award']

[1] Combined: 0.5219 (Vector: 0.2032, Graph: 1.0000)
    Entities: ['Nobel Prize', 'Turing Award', 'The Turing']
    The Turing Award is often called the Nobel Prize of computing. It was e

In [20]:
# Comparison Test 2: Relationship query
compare_rag_systems("What did Geoffrey Hinton work on?")


Query: What did Geoffrey Hinton work on?

📊 PLAIN RAG RESULTS:
--------------------------------------------------------------------------------

[1] Score: 0.3814
    Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. He is known for his work on artificial neural networks....

[2] Score: 0.3641
    He is known for his work on artificial neural networks. Hinton worked at the University of Toronto 
    and Google....

[3] Score: 0.1692
    The University of Toronto has been a major center for artificial intelligence research. Geoffrey Hinton established a research group there focusing on...


📊 GraphRAG RESULTS:
--------------------------------------------------------------------------------

🏷️ Query Entities Detected:
   PERSON: ['Geoffrey Hinton']

[1] Combined: 0.5289 (Vector: 0.3814, Graph: 0.7500)
    Entities: ['Geoffrey Hinton']
    Geoffrey Hinton is a British-Canadian cognitive psychologist and computer scientist. He is known for his work on a

In [21]:
# Comparison Test 3: Multi-hop query
compare_rag_systems("What technologies were developed by Turing Award winners?")


Query: What technologies were developed by Turing Award winners?

📊 PLAIN RAG RESULTS:
--------------------------------------------------------------------------------

[1] Score: 0.2475
    The award recognizes contributions of lasting 
    importance to computing. Winners receive a prize of one million dollars....

[2] Score: 0.1980
    The Turing Award is often called the Nobel Prize of computing. It was established in 1966 
    by the Association for Computing Machinery....

[3] Score: 0.1877
    He is a professor at New York University and works at Meta AI Research. LeCun developed 
    convolutional neural networks and received the Turing Awa...


📊 GraphRAG RESULTS:
--------------------------------------------------------------------------------

🏷️ Query Entities Detected:
   PERSON: ['Turing Award']
   AWARD: ['Turing Award']

[1] Combined: 0.5188 (Vector: 0.1980, Graph: 1.0000)
    Entities: ['Nobel Prize', 'Turing Award', 'The Turing']
    The Turing Award is often called t

### Key Observations

**When Plain RAG Works Well:**
- Simple keyword matching queries
- Direct information lookup
- Single-document information

**When GraphRAG Excels:**
- Entity-centric queries ("Tell me about X")
- Relationship queries ("How is X related to Y?")
- Multi-hop reasoning ("What did people who worked with X create?")
- Queries requiring connected context

<a id='section10'></a>
## 10. Exercises and Extensions

Now it's your turn! Here are hands-on exercises to deepen your understanding.

---

### Exercise 1: Stop Word Filtering (Easy)

**Goal**: Improve TF-IDF by filtering common words that don't add meaning.

**Tasks**:
1. Add a stop word list to the `TfidfVectorizer`
2. Filter stop words during tokenization
3. Compare vocabulary size before/after filtering

**Hints**:
- Common stop words: "the", "is", "at", "which", "on", "and", "a", "an"
- Modify the `tokenize()` method
- You can use Python's set for efficient lookup

In [22]:
# Exercise 1: Your code here

# TODO: Create an enhanced TfidfVectorizer with stop word filtering
class EnhancedTfidfVectorizer(TfidfVectorizer):
    def __init__(self):
        super().__init__()
        # TODO: Add stop words set
        self.stop_words = set([
            # Add your stop words here
        ])

    def tokenize(self, text: str) -> List[str]:
        # TODO: Modify to filter stop words
        tokens = re.findall(r'\b[a-z]+\b', text.lower())
        # Filter stop words here
        return tokens

# TODO: Test your implementation
# enhanced_vectorizer = EnhancedTfidfVectorizer()
# enhanced_vectors = enhanced_vectorizer.fit_transform(all_chunks)
# print(f"Original vocabulary size: {len(vectorizer.vocabulary)}")
# print(f"Enhanced vocabulary size: {len(enhanced_vectorizer.vocabulary)}")

### Exercise 2: Graph Visualization (Medium)

**Goal**: Create a visual representation of the knowledge graph.

**Tasks**:
1. Write a function to export the graph to a format suitable for visualization
2. Create a simple text-based visualization showing entity connections
3. (Bonus) Use a library like `networkx` or `matplotlib` for graphical visualization

**Hints**:
- Start with a small subgraph (e.g., one entity and its neighbors)
- Consider using ASCII art for text visualization
- Edge weights can be shown as line thickness or labels

In [23]:
# Exercise 2: Your code here

def visualize_entity_neighborhood(kg: KnowledgeGraph, entity: str, max_neighbors: int = 5):
    """
    Create a text-based visualization of an entity's neighborhood.

    TODO: Implement this function
    """
    pass

# TODO: Test your visualization
# visualize_entity_neighborhood(kg, "Geoffrey Hinton")

### Exercise 3: Improved Entity Extraction (Medium-Hard)

**Goal**: Enhance entity extraction with better pattern matching.

**Tasks**:
1. Add more entity types (e.g., TECHNOLOGY, LOCATION, DATE)
2. Implement pattern matching for these new types
3. Add co-reference resolution (e.g., "he" → "Geoffrey Hinton")
4. Evaluate improvement in entity coverage

**Hints**:
- Technology patterns: "neural networks", "deep learning", "CNN", "RNN"
- Use word context to identify entities
- For co-reference, track the last mentioned PERSON entity

In [24]:
# Exercise 3: Your code here

class ImprovedEntityExtractor(EntityExtractor):
    def __init__(self):
        super().__init__()
        # TODO: Add new entity patterns
        # TODO: Add co-reference tracking
        pass

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        # TODO: Implement improved extraction
        pass

# TODO: Test and compare with original extractor

### Exercise 4: Hybrid Scoring Function (Hard)

**Goal**: Experiment with different ways to combine vector and graph scores.

**Tasks**:
1. Implement at least 3 different scoring strategies:
   - Weighted average (already done)
   - Multiplicative (score = vector_score × graph_score)
   - Adaptive (weight changes based on query type)
2. Create an evaluation framework to compare strategies
3. Test each strategy on a set of queries
4. Analyze which strategy works best for which query types

**Hints**:
- Query types: entity-centric, relationship-based, concept-based
- Metrics: overlap with expected results, rank of best chunk
- Adaptive weighting could use query entity count

In [25]:
# Exercise 4: Your code here

class HybridScorer:
    """
    Experiment with different hybrid scoring strategies.
    """

    @staticmethod
    def weighted_average(vector_score, graph_score, alpha=0.6):
        # TODO: Already implemented in GraphRAG
        return alpha * vector_score + (1 - alpha) * graph_score

    @staticmethod
    def multiplicative(vector_score, graph_score):
        # TODO: Implement multiplicative scoring
        pass

    @staticmethod
    def adaptive(vector_score, graph_score, query_entity_count):
        # TODO: Implement adaptive weighting based on query characteristics
        pass

# TODO: Create evaluation framework and test

### Exercise 5: Multi-hop Reasoning Paths (Hard)

**Goal**: Not just retrieve chunks, but explain the reasoning path through the graph.

**Tasks**:
1. Implement a function to find paths between query entities and retrieved chunks
2. Generate natural language explanations of these paths
3. Rank paths by relevance
4. Create visualizations of reasoning chains

**Example**:
```
Query: "What did people who worked with Hinton create?"

Reasoning Path:
Geoffrey Hinton → (co_occurs) → Yoshua Bengio → (developed) → Deep Learning
                                                              → Neural Networks
```

**Hints**:
- Use breadth-first search (BFS) to find shortest paths
- Store path metadata during graph traversal
- Consider path diversity (don't show redundant paths)

In [26]:
# Exercise 5: Your code here

def find_reasoning_paths(kg: KnowledgeGraph, start_entity: str, target_entities: List[str], max_depth: int = 3):
    """
    Find reasoning paths from start entity to target entities.

    Returns:
    --------
    List of paths, where each path is a list of (entity, relation) tuples

    TODO: Implement BFS-based path finding
    """
    pass

def explain_path(path):
    """
    Convert a path to natural language.

    TODO: Implement path explanation
    """
    pass

# TODO: Test your implementation

### Exercise 6: Add Your Own Documents (Open-ended)

**Goal**: Apply GraphRAG to a domain of your choice.

**Tasks**:
1. Create a corpus of 5-10 documents about a topic you're interested in
2. Run the entire GraphRAG pipeline on your documents
3. Design queries that test GraphRAG's capabilities
4. Analyze what works and what doesn't
5. Propose improvements specific to your domain

**Suggested Domains**:
- Movie plots and actors
- Scientific papers and authors
- Company relationships and products
- Historical events and figures
- Sports teams and players

In [27]:
# Exercise 6: Your code here

# TODO: Define your custom documents
custom_documents = [
    # Add your documents here
]

# TODO: Run the pipeline
# 1. Chunk documents
# 2. Vectorize
# 3. Extract entities
# 4. Build graph
# 5. Test queries

# TODO: Analyze and document your findings



## Summary and Key Takeaways

### GraphRAG Advantages

✅ **Better Context**: Retrieves related information through entity connections  
✅ **Multi-hop Reasoning**: Can traverse relationships to find distant but relevant information  
✅ **Entity Awareness**: Understands that different mentions refer to same entities  
✅ **Relationship Modeling**: Captures how concepts relate to each other


### Next Steps

To build production GraphRAG systems:
- Use neural entity extractors (spaCy, Hugging Face)
- Implement vector databases (FAISS, Pinecone)
- Add graph databases (Neo4j, Neptune)
- Integrate with LLMs (OpenAI, Claude, Llama)
- Implement caching and optimization

### Resources for Further Learning

- **Papers**: "Graph Neural Networks", "REALM", "RAG: Retrieval-Augmented Generation"
- **Libraries**: LangChain, LlamaIndex, Haystack
- **Courses**: Stanford CS224W (Graph ML), fast.ai

---

## Congratulations! 🎉
**Keep exploring and building!** 🚀